In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# !pip install "transformers[torch]"
# !pip install peft
# !pip install evaluate

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model
from sklearn.utils.class_weight import compute_class_weight
import evaluate
from datasets import Dataset, ClassLabel, load_from_disk

In [ ]:
# -------------
# CONFIGURATION
# -------------
HDFS_FILE_PATH = "data/NPZ/HDFS.npz"
ANOMALY_LABEL_FILE_PATH = "data/CSV_100K/anomaly_label.csv"
HDFS_TEMPLATES_FILE_PATH = "data/HDFS_templates.csv"
TOKENIZED_DATASET_PATH = "tokenized_dataset"
# CSV_EVENTS = 0,
# NPZ_EVENTS = 1,
# NPZ_EVENT_IDS = 2
# TOKENIZED_DATASET = 3
TYPE_OF_DATA = 3

# TRAINING ARGUMENTS
NUM_TRAIN_EPOCHS = 5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 5e-5

# LORA CONFIGURATION
LORA_RANK = 64
LORA_ALPHA = 32
LORA_DROPOUT = 0.1

# WEIGHTED TRAINER
CLASS_WEIGHTS_ENABLED = None
FOCAL_ENABLED = True
GAMMA = 2.0

# MODEL
MODEL_NAME = "bert-base-uncased"

In [ ]:
# --------
# LOAD CSV
# --------
def extract_block(content):
    if "blk_" in str(content):
        start = content.find("blk_")
        part = content[start:].split()[0].strip(',')
        return part
    return np.nan

def load_csv(path_logs, path_labels):
    logs = pd.read_csv(path_logs)
    labels = pd.read_csv(path_labels)

    logs['BlockId'] = logs['Content'].apply(extract_block)
    logs = logs.dropna(subset=["BlockId"])

    block_seqs = logs.groupby("BlockId")["EventTemplate"].apply(list)
    labels = labels.set_index("BlockId")["label"]
    label2id = {"Normal": 0, "Anomaly": 1}

    df = pd.DataFrame({
        "text": block_seqs,
        "label": labels
    }).dropna().reset_index(drop=True)

    df["label"] = df["label"].map(label2id).astype(int)

    df["text"] = df["text"].apply(
        lambda seq: ', '.join(seq) if isinstance(seq, (list, np.ndarray)) else str(seq)
    )
    return df

In [ ]:
# --------
# LOAD NPZ
# --------
def get_events_dict(path):
    event_templates = pd.read_csv(path)
    return dict(zip(event_templates['EventId'], event_templates['EventTemplate']))

def load_npz(path_npz, path_event_templates=None):
    data = np.load(path_npz, allow_pickle=True)
    x_train = data['x_data']
    y_train = data['y_data']

    df = pd.DataFrame({
        'text': x_train,
        'label': y_train
    })
    if path_event_templates:
        eventid2event = get_events_dict(path_event_templates)
        df['text'] = df['text'].apply(lambda seq: ', '.join([eventid2event[event_id] for event_id in seq]))
    else:
        df['text'] = df['text'].apply(lambda seq: ', '.join(seq))

    return df

In [ ]:
# -------------------------------
# HUGGINGFACE DATASET PREPARATION
# -------------------------------
def create_hg_split_dataset(dataset_pd):
    dataset = Dataset.from_pandas(dataset_pd)
    dataset = dataset.cast_column(
        "label",
        ClassLabel(num_classes=2, names=["0", "1"])
    )
    split_dataset = dataset.train_test_split(
        test_size=0.2,
        stratify_by_column="label",
        seed=42
    )
    return split_dataset

In [ ]:
# ------------
# TOKENIZATION
# ------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

def tokenize_dataset(split_dataset):
    tokenized_dataset = split_dataset.map(
        tokenize_function,
        batched=True,
        num_proc=4,
        remove_columns=["text"]
    )

    tokenized_dataset.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "label"]
    )
    return tokenized_dataset



In [ ]:
if TYPE_OF_DATA == 0:
    dataset_pd = load_csv(HDFS_FILE_PATH, ANOMALY_LABEL_FILE_PATH)
    dataset = create_hg_split_dataset(dataset_pd)
    tokenized_dataset = tokenize_dataset(dataset)
    tokenized_dataset.save_to_disk(TOKENIZED_DATASET_PATH)
elif TYPE_OF_DATA == 1:
    dataset_pd = load_npz(HDFS_FILE_PATH)
    dataset = create_hg_split_dataset(dataset_pd)
    tokenized_dataset = tokenize_dataset(dataset)
    tokenized_dataset.save_to_disk(TOKENIZED_DATASET_PATH)
elif TYPE_OF_DATA == 2:
    dataset_pd = load_npz(HDFS_FILE_PATH, HDFS_TEMPLATES_FILE_PATH)
    dataset = create_hg_split_dataset(dataset_pd)
    tokenized_dataset = tokenize_dataset(dataset)
    tokenized_dataset.save_to_disk(TOKENIZED_DATASET_PATH)
elif TYPE_OF_DATA == 3:
    tokenized_dataset = load_from_disk(TOKENIZED_DATASET_PATH)
else:
    raise ValueError("Invalid TYPE_OF_DATA value")

In [ ]:
tokenized_dataset['train']

In [ ]:
# ----------------
# TRAIN-TEST SPLIT
# ----------------
train_ds, test_ds = tokenized_dataset["train"], tokenized_dataset["test"]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "Normal", 1: "Anomaly"},
    label2id={"Normal": 0, "Anomaly": 1},
)

In [ ]:
# --------------
# TRAINING SETUP
# --------------
args = TrainingArguments(
    output_dir="./bert_hdfs_lora",
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    bf16=True,
)

lora_cfg = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["query", "value"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="SEQ_CLS",
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision.compute(predictions=preds, references=labels, average="binary")["precision"],
        "recall": recall.compute(predictions=preds, references=labels, average="binary")["recall"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

In [ ]:
class CustomTrainer(Trainer):
    def __init__(self, class_weights=None, focal=False, gamma=2.0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.focal = focal
        self.gamma = gamma

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs[0]

        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        ce_loss = torch.nn.functional.cross_entropy(logits, labels, weight=weight, reduction="none")

        if self.focal:
            pt = torch.exp(-ce_loss)
            loss = ((1 - pt) ** self.gamma * ce_loss).mean()
        else:
            loss = ce_loss.mean()

        return (loss, outputs) if return_outputs else loss

In [ ]:
labels = np.array(train_ds['label'])
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)
print(weights)
class_weights = torch.tensor(weights, dtype=torch.float) if CLASS_WEIGHTS_ENABLED else None
print("Class weights:", class_weights)


In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
trainer = CustomTrainer(
    model=model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu")),
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    focal=FOCAL_ENABLED,
    gamma=GAMMA
)

In [ ]:
trainer.train()

In [ ]:
# adapter_dir = os.path.join("output", "lora_adapter")
# trainer.model.save_pretrained(adapter_dir)
# tokenizer.save_pretrained(adapter_dir)
# print(f"\nLoRA adapter saved to: {adapter_dir}")

In [ ]:
# def predict_block(log_templates):
#     text = " [SEP] ".join(log_templates)
#     enc = tokenizer(text, truncation=True, max_length=256, return_tensors="pt").to(trainer.model.device)
#     with torch.no_grad():
#         out = trainer.model(**enc)
#         probs = torch.softmax(out.logits, dim=-1).squeeze(0).tolist()
#     pred_id = int(torch.tensor(probs).argmax().item())
#     return {"pred_label": model.config.id2label[pred_id], "probs": {model.config.id2label[i]: float(p) for i, p in enumerate(probs)}}



# example_seq = anomaly_row = eval_df[eval_df["LabelId"] == 1].iloc[0]["text"]
# print("Example prediction:", predict_block(example_seq))

In [ ]:
# eval_df[eval_df["LabelId"] == 1].head()

In [ ]:
# anomaly_row = eval_df[eval_df["LabelId"] == 1].iloc[0]["text"]
# print(anomaly_row)

# example_seq = eval_df.iloc[3]["text"]
# print(example_seq)